# Демо: ошибки и модули

Финальное демо первой недели. Пройдём `try / except / else / finally`, научимся поднимать свои исключения через `raise`, посмотрим формы `import` и разберём, что значит `ModuleNotFoundError`. Прокликаем Shift+Enter — ошибки внутри ячеек обёрнуты в `try / except`, чтобы ноутбук исполнился до конца без падения.

## Часть 1. `try` / `except`

Сейчас посмотрим базовый перехват. Шаблон: `try` — код, который может упасть; `except <Класс>` — код, который выполнится, если конкретное исключение поднялось. Если исключение в `try` НЕ поднялось, блок `except` пропускается.

In [1]:
# ValueError — int() не смог распарсить строку
try:
    result = int('не число')
except ValueError as e:
    print(f'ValueError: {e}')   # invalid literal for int() with base 10: ...
    result = 0

print(f'result = {result}')      # 0 — программа продолжила работу

ValueError: invalid literal for int() with base 10: 'не число'
result = 0


In [2]:
# Один try может ловить несколько разных классов
user_input = '10'
divisor = '0'

try:
    answer = int(user_input) / int(divisor)
except ValueError:
    answer = None                # не удалось распарсить
except ZeroDivisionError:
    answer = float('inf')        # делили на ноль

print(answer)                    # inf

inf


**Подводный камень:** `except:` без класса (или `except Exception:`) ловит вообще всё — включая опечатки в именах, прерывание `Ctrl+C`, ошибки импорта. Из-за этого настоящие баги маскируются под «обработанную ошибку», и понять, что сломалось, становится тяжело. Лови конкретные классы — `ValueError`, `KeyError`, `TypeError` — а `Exception` только там, где осознанно хочется широкий перехват.

In [3]:
# А что если использовать broad except — он поглотит даже NameError
try:
    print(opechatka_v_imeni)     # переменной нет — NameError
except Exception as e:
    print(f'поймали: {type(e).__name__}: {e}')
    # NameError проглочен, и без `as e` мы бы даже не узнали, что это была опечатка

поймали: NameError: name 'opechatka_v_imeni' is not defined


## Часть 2. `else` и `finally`

У `try` есть ещё две ветки: `else` (выполняется, если исключения НЕ было) и `finally` (выполняется ВСЕГДА — и при успехе, и при ошибке). Шаблон-целиком: `try → except → else → finally`.

In [4]:
# Успешный путь: try прошёл → else сработал → finally сработал
try:
    value = int('42')
except ValueError:
    print('except: не сработал — исключения не было')
else:
    print(f'else: всё ок, value = {value}')
finally:
    print('finally: выполнюсь в любом случае')

else: всё ок, value = 42
finally: выполнюсь в любом случае


In [5]:
# Путь с ошибкой: try упал → except сработал → else пропущен → finally сработал
try:
    value = int('не число')
except ValueError as e:
    print(f'except: поймали {e}')
else:
    print('else: пропустится — исключение было')
finally:
    print('finally: всё равно выполнюсь')

except: поймали invalid literal for int() with base 10: 'не число'
finally: всё равно выполнюсь


Типичный паттерн `finally` — освобождение ресурса (закрыть файл, сетевое соединение, транзакцию БД), который должен быть отпущен независимо от того, упала операция или прошла.

## Часть 3. `raise` — поднять исключение самому

`raise <Класс>(сообщение)` — поднять исключение вручную. Используется, когда хотите сигнализировать о невалидных данных. Стандартные классы (`ValueError`, `TypeError`, `KeyError`) покрывают большинство случаев.

In [6]:
# Поднимаем ValueError на невалидном делителе
a, b = 10, 0

try:
    if b == 0:
        raise ValueError('делитель не может быть нулём')
    result = a / b
except ValueError as e:
    print(f'ValueError: {e}')
    result = None

print(f'result = {result}')      # None — деление не выполнилось

ValueError: делитель не может быть нулём
result = None


Зачем `raise`, если можно просто проверить условие через `if/else`? Потому что исключение **прерывает поток** и поднимается вверх по стеку, пока его кто-то не поймает. Это удобно, когда невалидные данные глубоко внутри логики — не приходится тащить «коды ошибок» через каждый шаг.

In [7]:
# Несколько разных проверок — каждая поднимает свой ValueError
name = ''
age = -3

try:
    if not name:
        raise ValueError('имя не может быть пустым')
    if age < 0:
        raise ValueError(f'возраст не может быть отрицательным: {age}')
    print('всё ок')
except ValueError as e:
    print(f'ошибка валидации: {e}')
# первый же raise остановит выполнение и улетит в except

ошибка валидации: имя не может быть пустым


**Подводный камень:** не пиши `raise` без сообщения. `raise ValueError` без скобок и текста — синтаксически валидно, но при отладке ты увидишь только имя класса без контекста, что именно сломалось. Всегда передавай осмысленное сообщение.

In [8]:
# Имитация прокидывания исключения «снизу вверх» через try
raw = '3,4,abc'

try:
    parts = raw.split(',')
    if len(parts) != 2:
        raise ValueError(f'ожидалось 2 числа, получено {len(parts)}')
    x, y = int(parts[0]), int(parts[1])
    print(f'x={x}, y={y}')
except ValueError as e:
    print(f'поймали: {e}')
# raise из внешнего if поймается тем же except — исключение «всплыло»

поймали: ожидалось 2 числа, получено 3


## Часть 4. `import` — четыре формы

Сейчас посмотрим четыре способа подключить модуль. Все четыре встретятся в чужом коде; пишут чаще всего две: `import X` и `import X as Y`.

In [9]:
# Форма 1: import целиком — доступ через math.pi, math.sqrt
import math

print(math.pi)                   # 3.141592653589793
print(math.sqrt(16))             # 4.0
print(math.floor(3.7))           # 3

3.141592653589793
4.0
3


In [10]:
# Форма 2: from X import name — короче, но не видно происхождения
from math import pi, sqrt

print(pi)                        # 3.141592653589793
print(sqrt(25))                  # 5.0

3.141592653589793
5.0


In [11]:
# Форма 3: import X as Y — переименование, стандарт для numpy/pandas
import math as m

print(m.pi)                      # 3.141592653589793
print(m.ceil(3.2))               # 4

3.141592653589793
4


In [12]:
# Форма 4: from X import name as alias — редкая, при конфликте имён
from datetime import datetime as DT

print(DT.now().year)             # текущий год

2026


**Подводный камень:** `from module import *` — анти-паттерн. Эта форма тащит в текущий файл *все* имена модуля, и через месяц никто не понимает, откуда взялась `chain` или `product`. В промышленном коде запрещают линтерами.

### Гард `if __name__ == '__main__'`

Когда Python импортирует ваш файл, он его **исполняет** один раз сверху вниз. Если в файле на верхнем уровне лежит код «запусти приложение» — он выполнится при импорте, чего обычно не хотят. Гард `if __name__ == '__main__':` решает это: блок внутри гарда исполняется только при прямом запуске (`python script.py`), но НЕ при импорте.

In [13]:
# В скрипте обычно: import-ы → определения функций → if __name__ == '__main__': main()
# Здесь имитируем — внутри ноутбука __name__ равен '__main__'
print(f'__name__ = {__name__!r}')   # '__main__' — мы запустили этот код напрямую

if __name__ == '__main__':
    print('блок запустился — это прямой запуск')
else:
    print('блок пропустился — нас импортировали')

__name__ = '__main__'
блок запустился — это прямой запуск


## Часть 5. `ModuleNotFoundError`

Если Python не нашёл модуль — поднимает `ModuleNotFoundError` (подкласс `ImportError`). Причины: опечатка в имени, пакет не установлен в текущем окружении, файл лежит в другой папке. Оборачиваем в `try / except`, чтобы посмотреть текст ошибки.

In [14]:
# Имитируем опечатку или отсутствие пакета
try:
    import some_nonexistent_module
except ModuleNotFoundError as e:
    print(f'ModuleNotFoundError: {e}')

ModuleNotFoundError: No module named 'some_nonexistent_module'


Куда Python смотрит при `import X` — видно в `sys.path`. Это список путей: текущая папка, стандартная библиотека, установленные пакеты (`site-packages`). Если модуля нет ни в одном из этих путей — `ModuleNotFoundError`.

In [15]:
# Посмотрим первые 5 путей, где Python ищет модули
import sys

for path in sys.path[:5]:
    print(path)

/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python39.zip
/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9
/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/lib-dynload

/Users/syakubson/Library/Python/3.9/lib/python/site-packages


## Мини-задания

Три упражнения. Подсказок к методам нет — вспомни сам. Все задания решаются inline (без `def` и `class`) — на этой неделе функции и классы ещё не разбирали.

**Задание 1.** Дано `s = '42'`. В переменной `result` сохрани `int(s)`, если строка парсится в число; если ловится `ValueError` — присвой `result = 'неверное число'`. Проверь на двух разных входах: сначала `s = '42'`, потом `s = 'abc'`. Используй `try / except`, без `def`.

**Задание 2.** Даны `balance = 100` и `amount = 150`. Если `amount > balance`, подними `ValueError('недостаточно средств')` и поймай его во внешнем `try / except`, выведя текст ошибки. Иначе выведи новый баланс `balance - amount`. Проверь на двух парах: `(100, 30)` и `(50, 100)`.

**Задание 3.** Разбор строки в кортеж — стандартный способ вернуть из парсинга сразу пару значений (координаты, диапазоны, пары ключ-значение). Дано `s = '3,4'`. Получи кортеж `(3, 4)` — раздели строку по запятой и сконвертируй обе части в `int`. Оберни весь код в `try / except ValueError` — если формат неверный, в `result` положи `None`. Проверь на `'3,4'` и на `'3,abc'`.

In [16]:
# Задание 1: безопасный парсинг строки в число
s = '42'
# Твой код — заполни result через try / except:

# print(result)        # ожидается 42

# Повтори с s = 'abc' — ожидается строка 'неверное число'

In [17]:
# Задание 2: проверка баланса с raise + try / except
balance = 100
amount = 150
# Твой код — внутри try подними ValueError, в except выведи сообщение:

# Повтори с balance = 100, amount = 30 — ожидается, что выведется новый баланс 70

In [18]:
# Задание 3: парсинг пары чисел из строки
s = '3,4'
result = None
# Твой код — раздели s по запятой, сконвертируй обе части в int,
# собери в кортеж и положи в result. При ValueError оставь None:

# print(result)        # ожидается (3, 4)

# Повтори с s = '3,abc' — ожидается None